# 06 - Entrenamiento Transformer para detección de platos

## Objetivo

Este notebook entrena el primer modelo NER de Hidden Gems para detectar menciones de platos en reseñas gastronómicas.

El modelo recibirá una review tokenizada y deberá clasificar cada token como:

- `O`: fuera de una entidad.
- `B-DISH`: inicio de una mención de plato.
- `I-DISH`: continuación de una mención de plato.

Ejemplo:

```text
The crab legs were amazing
O   B-DISH I-DISH O    O
```

El entrenamiento parte del dataset BIO generado en el Notebook 05.

##Estrategia

Primero se ejecutará una prueba rápida con `dish_ner_bio_smoke_test.jsonl`.

Si el flujo funciona correctamente, se entrenará con el dataset completo:

`dish_ner_bio_high_precision_with_negatives.jsonl`

In [31]:
# ============================================================
# 01. Instalación de dependencias
# ============================================================

!pip -q install transformers datasets evaluate accelerate seqeval
!pip -q install pandas numpy scikit-learn matplotlib tqdm

In [32]:
# ============================================================
# 02. Imports y configuración base
# ============================================================

from pathlib import Path
import json
import random
import os
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm

import torch

from datasets import Dataset, DatasetDict

from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 300)

print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cu128
CUDA disponible: True
GPU: Tesla T4


In [33]:
# ============================================================
# 03. Detectar entorno y preparar carpetas
# ============================================================

IS_KAGGLE = Path("/kaggle/input").exists()
IS_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in globals() else False

if IS_KAGGLE:
    ENV_NAME = "kaggle"
elif IS_COLAB:
    ENV_NAME = "colab"
else:
    ENV_NAME = "local"

print("Entorno detectado:", ENV_NAME)

if IS_KAGGLE:
    INPUT_DIR = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    PROJECT_DIR = WORKING_DIR / "hidden_gems_ai"

elif IS_COLAB:
    from google.colab import drive

    try:
        !fusermount -u /content/drive 2>/dev/null || true
        !rm -rf /content/drive

        drive.mount("/content/drive", force_remount=True, timeout_ms=120000)

        PROJECT_DIR = Path("/content/drive/MyDrive/hidden_gems_ai")

    except Exception as e:
        print("No se ha podido montar Drive. Se usará almacenamiento temporal.")
        print("Error:", e)

        PROJECT_DIR = Path("/content/hidden_gems_ai")

else:
    PROJECT_DIR = Path.cwd()

OUTPUT_DIR = PROJECT_DIR / "outputs" / "dish_ner_transformer"
MODEL_DIR = OUTPUT_DIR / "model"
METRICS_DIR = OUTPUT_DIR / "metrics"
PREDICTIONS_DIR = OUTPUT_DIR / "predictions"
ARTIFACTS_DIR = OUTPUT_DIR / "artifacts"

for path in [PROJECT_DIR, OUTPUT_DIR, MODEL_DIR, METRICS_DIR, PREDICTIONS_DIR, ARTIFACTS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

Entorno detectado: kaggle
PROJECT_DIR: /kaggle/working/hidden_gems_ai
OUTPUT_DIR: /kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer


In [34]:
# ============================================================
# 04. Diagnóstico de archivos disponibles
# ============================================================

if IS_KAGGLE:
    print("Archivos disponibles en /kaggle/input:")

    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))
else:
    print("No estás en Kaggle. Se buscarán archivos en el proyecto local.")

Archivos disponibles en /kaggle/input:
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_labels.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_manual_review_sample_v2_150.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/transformer_sentiment_metrics.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidates_v2.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_detection_exploration_summary_v2.json
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/yelp_translation_gold_seed_300_es_ready_v1.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_smoke_test.jsonl
/kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/weak_dish_candidate_frequencies_v2.csv
/kaggle/input/datasets/ivanarteagacordero/hidden-

In [35]:
# ============================================================
# 05. Localizar datasets BIO y labels
# ============================================================

REQUIRED_FILES = {
    "bio_full": "dish_ner_bio_high_precision_with_negatives.jsonl",
    "bio_smoke": "dish_ner_bio_smoke_test.jsonl",
    "labels": "dish_ner_labels.json",
}

def find_file(filename: str) -> Path:
    candidate_paths = []

    if IS_KAGGLE:
        candidate_paths.extend(list(Path("/kaggle/input").rglob(filename)))

    candidate_paths.extend(list(PROJECT_DIR.rglob(filename)))
    candidate_paths.extend(list(Path.cwd().rglob(filename)))

    candidate_paths = list(dict.fromkeys(candidate_paths))

    if not candidate_paths:
        raise FileNotFoundError(
            f"No se ha encontrado el archivo requerido: {filename}\n"
            "En Kaggle, asegúrate de haberlo añadido desde Add Data."
        )

    return candidate_paths[0]


BIO_FULL_PATH = find_file(REQUIRED_FILES["bio_full"])
BIO_SMOKE_PATH = find_file(REQUIRED_FILES["bio_smoke"])
LABELS_PATH = find_file(REQUIRED_FILES["labels"])

print("BIO_FULL_PATH:", BIO_FULL_PATH)
print("BIO_SMOKE_PATH:", BIO_SMOKE_PATH)
print("LABELS_PATH:", LABELS_PATH)

BIO_FULL_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
BIO_SMOKE_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_smoke_test.jsonl
LABELS_PATH: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_labels.json


In [36]:
# ============================================================
# 06. Configuración inicial del entrenamiento
# ============================================================

# Primero lo dejamos en True para probar que todo el flujo funciona.
# Cuando todo vaya bien, cambia a False para usar el dataset completo.
USE_SMOKE_TEST = False

if USE_SMOKE_TEST:
    BIO_DATASET_PATH = BIO_SMOKE_PATH
else:
    BIO_DATASET_PATH = BIO_FULL_PATH

MODEL_CHECKPOINT = "distilbert-base-uncased"

MAX_LENGTH = 256

NUM_EPOCHS = 2
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32

# Para NER suele ser mejor no etiquetar todos los subtokens.
# Solo el primer subtoken de cada palabra recibe etiqueta; el resto será -100.
LABEL_ALL_TOKENS = False

print("Dataset seleccionado:", BIO_DATASET_PATH)
print("USE_SMOKE_TEST:", USE_SMOKE_TEST)
print("Modelo:", MODEL_CHECKPOINT)
print("MAX_LENGTH:", MAX_LENGTH)

Dataset seleccionado: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-yelp-nlp/dish_ner_bio_high_precision_with_negatives.jsonl
USE_SMOKE_TEST: False
Modelo: distilbert-base-uncased
MAX_LENGTH: 256


In [37]:
# ============================================================
# 07. Funciones de carga JSONL
# ============================================================

def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    invalid_json_count = 0

    with path.open("r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Leyendo {path.name}"):
            line = line.strip()

            if not line:
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                invalid_json_count += 1

    print(f"Archivo: {path.name}")
    print(f"Registros cargados: {len(records):,}")
    print(f"Líneas JSON inválidas: {invalid_json_count:,}")

    return pd.DataFrame(records)


bio_df = load_jsonl(BIO_DATASET_PATH)

with LABELS_PATH.open("r", encoding="utf-8") as f:
    labels_config = json.load(f)

NER_LABELS = labels_config["labels"]
label2id = {str(k): int(v) for k, v in labels_config["label2id"].items()}
id2label = {int(k): str(v) for k, v in labels_config["id2label"].items()}

print("Shape BIO:", bio_df.shape)
print("Labels:", NER_LABELS)
print("label2id:", label2id)
print("id2label:", id2label)

display(bio_df.head(3))

Leyendo dish_ner_bio_high_precision_with_negatives.jsonl: 0it [00:00, ?it/s]

Archivo: dish_ner_bio_high_precision_with_negatives.jsonl
Registros cargados: 43,161
Líneas JSON inválidas: 0
Shape BIO: (43161, 18)
Labels: ['O', 'B-DISH', 'I-DISH']
label2id: {'O': 0, 'B-DISH': 1, 'I-DISH': 2}
id2label: {0: 'O', 1: 'B-DISH', 2: 'I-DISH'}


,corpus_document_id,review_id,business_id,split,rating_value,sentiment_label,text,tokens,token_offsets,ner_tags,ner_tag_names,entities,entity_count,token_count,dish_token_count,dataset_name,labeling_method,human_review_status
0,yelp_ce585eabced3468556e19c1a,RAhYHS028TKwAai7eFM4rA,ZWRgvtyHRobwpi4G-Z4STQ,test,5.0,positive,Since going paleo it's hard to find something that doesn't taste like cardboard to satisfy my pizza cravings. The gluten free pizza here is the best I've found and they even personalize it any way I want including no cheese and extra sauce. It's my new fav!,"[Since, going, paleo, it's, hard, to, find, something, that, doesn't, taste, like, cardboard, to, satisfy, my, pizza, cravings, ., The, gluten, free, pizza, here, is, the, best, I've, found, and, they, even, personalize, it, any, way, I, want, including, no, cheese, and, extra, sauce, ., It's, m...","[{'start': 0, 'end': 5}, {'start': 6, 'end': 11}, {'start': 12, 'end': 17}, {'start': 18, 'end': 22}, {'start': 23, 'end': 27}, {'start': 28, 'end': 30}, {'start': 31, 'end': 35}, {'start': 36, 'end': 45}, {'start': 46, 'end': 50}, {'start': 51, 'end': 58}, {'start': 59, 'end': 64}, {'start': 65...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[{'start': 94, 'end': 99, 'label': 'DISH', 'text': 'pizza', 'normalized_text': 'pizza', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 126, 'end': 131, 'label': 'DISH', 'text': 'pizza', 'normalized_text': 'pizza', 'candidate_gr...",2,50,2,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
1,yelp_3e0c6444441a9272c32f30ce,pKcQBSuGskizQ992yCRJTg,5iuo1kvv0XZMS0bUOoLz2Q,test,2.0,negative,"Wasn't too impressed with their food. Ordered the tuna tartare, which never came out :(. Had the sirloin steak, which wasn't even that good, and it was also overcooked. Fortunately I was with good company, so the food didn't bother me as much. However, I probably won't come back and have the foo...","[Wasn't, too, impressed, with, their, food, ., Ordered, the, tuna, tartare, ,, which, never, came, out, :, (, ., Had, the, sirloin, steak, ,, which, wasn't, even, that, good, ,, and, it, was, also, overcooked, ., Fortunately, I, was, with, good, company, ,, so, the, food, didn't, bother, me, as,...","[{'start': 0, 'end': 6}, {'start': 7, 'end': 10}, {'start': 11, 'end': 20}, {'start': 21, 'end': 25}, {'start': 26, 'end': 31}, {'start': 32, 'end': 36}, {'start': 36, 'end': 37}, {'start': 38, 'end': 45}, {'start': 46, 'end': 49}, {'start': 50, 'end': 54}, {'start': 55, 'end': 62}, {'start': 62...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[{'start': 50, 'end': 54, 'label': 'DISH', 'text': 'tuna', 'normalized_text': 'tuna', 'candidate_granularity': 'dish_family', 'candidate_sources': ['dictionary'], 'weak_confidence_v2': 0.75}, {'start': 105, 'end': 110, 'label': 'DISH', 'text': 'steak', 'normalized_text': 'steak', 'candidate_gran...",2,66,2,dish_ner_weak_high_precision,weak_exact_match_v2,not_reviewed
2,yelp_ab3d21e21cc2584a47ab6510,aOb_YG545JLGJWPiEEY1xw,2Q1R2OhBbAQ581vK_r7NhA,train,5.0,positive,"Came here on a Monday night with a few friends. I don't know the area well, but I'm glad we decided on this place. We were seated quickly, and not even a little grumbling that they were closing in 45 minutes. The sushi was fresh. The food was in 

In [38]:
# ============================================================
# 08. Validación del dataset BIO cargado
# ============================================================

required_cols = [
    "review_id",
    "split",
    "text",
    "tokens",
    "ner_tags",
    "ner_tag_names",
    "entity_count",
]

missing_cols = [col for col in required_cols if col not in bio_df.columns]

if missing_cols:
    raise ValueError(f"Faltan columnas obligatorias: {missing_cols}")

valid_splits = {"train", "validation", "test"}

bio_df["split"] = bio_df["split"].astype(str).str.lower().str.strip()
bio_df = bio_df[bio_df["split"].isin(valid_splits)].copy()

def validate_row_lengths(row):
    return (
        isinstance(row["tokens"], list)
        and isinstance(row["ner_tags"], list)
        and len(row["tokens"]) == len(row["ner_tags"])
    )

bio_df["valid_lengths"] = bio_df.apply(validate_row_lengths, axis=1)

invalid_length_count = int((~bio_df["valid_lengths"]).sum())

if invalid_length_count > 0:
    raise ValueError(f"Hay {invalid_length_count} filas con tokens/ner_tags de distinta longitud.")

print("Dataset BIO validado correctamente.")
print("Total documentos:", len(bio_df))

print("\nSplit counts:")
display(bio_df["split"].value_counts())

print("\nEntity count summary:")
display(bio_df["entity_count"].describe())

print("\nDocs con/sin entidades:")
display(
    bio_df.assign(has_dish=bio_df["entity_count"] > 0)
    .groupby(["split", "has_dish"])
    .size()
    .reset_index(name="count")
)

Dataset BIO validado correctamente.
Total documentos: 43161

Split counts:


split
train         34182
validation     4502
test           4477
Name: count, dtype: int64


Entity count summary:


count    43161.000000
mean         2.274229
std          2.041167
min          0.000000
25%          1.000000
50%          2.000000
75%          3.000000
max         37.000000
Name: entity_count, dtype: float64


Docs con/sin entidades:


,split,has_dish,count
0,test,False,407
1,test,True,4070
2,train,False,1500
3,train,True,32682
4,validation,False,409
5,validation,True,4093


In [39]:
# ============================================================
# 09. Crear DatasetDict de Hugging Face
# ============================================================

hf_columns = [
    "review_id",
    "split",
    "tokens",
    "ner_tags",
    "ner_tag_names",
    "entity_count",
]

for col in hf_columns:
    if col not in bio_df.columns:
        bio_df[col] = None

train_pd = bio_df[bio_df["split"] == "train"][hf_columns].reset_index(drop=True)
val_pd = bio_df[bio_df["split"] == "validation"][hf_columns].reset_index(drop=True)
test_pd = bio_df[bio_df["split"] == "test"][hf_columns].reset_index(drop=True)

dataset = DatasetDict({
    "train": Dataset.from_pandas(train_pd, preserve_index=False),
    "validation": Dataset.from_pandas(val_pd, preserve_index=False),
    "test": Dataset.from_pandas(test_pd, preserve_index=False),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['review_id', 'split', 'tokens', 'ner_tags', 'ner_tag_names', 'entity_count'],
        num_rows: 34182
    })
    validation: Dataset({
        features: ['review_id', 'split', 'tokens', 'ner_tags', 'ner_tag_names', 'entity_count'],
        num_rows: 4502
    })
    test: Dataset({
        features: ['review_id', 'split', 'tokens', 'ner_tags', 'ner_tag_names', 'entity_count'],
        num_rows: 4477
    })
})

In [40]:
# ============================================================
# 10. Cargar tokenizer
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

print("Tokenizer cargado:", MODEL_CHECKPOINT)
print("Fast tokenizer:", tokenizer.is_fast)

Tokenizer cargado: distilbert-base-uncased
Fast tokenizer: True


In [41]:
# ============================================================
# 11. Alineación de etiquetas con subtokens
# ============================================================

def align_labels_with_tokens(labels, word_ids):
    """
    Alinea etiquetas BIO de palabras con subtokens del tokenizer.

    Si LABEL_ALL_TOKENS = False:
    - solo el primer subtoken recibe la etiqueta;
    - subtokens posteriores reciben -100 para ignorarlos en la loss.

    Si LABEL_ALL_TOKENS = True:
    - los subtokens posteriores heredan la etiqueta;
    - B-DISH puede convertirse en I-DISH en subtokens internos.
    """

    new_labels = []
    previous_word_id = None

    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)

        elif word_id != previous_word_id:
            new_labels.append(labels[word_id])

        else:
            if LABEL_ALL_TOKENS:
                label = labels[word_id]

                if label == label2id["B-DISH"]:
                    label = label2id["I-DISH"]

                new_labels.append(label)
            else:
                new_labels.append(-100)

        previous_word_id = word_id

    return new_labels


def tokenize_and_align_labels(batch):
    tokenized_inputs = tokenizer(
        batch["tokens"],
        truncation=True,
        is_split_into_words=True,
        max_length=MAX_LENGTH,
    )

    all_labels = batch["ner_tags"]
    aligned_labels = []

    for i, labels in enumerate(all_labels):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels.append(align_labels_with_tokens(labels, word_ids))

    tokenized_inputs["labels"] = aligned_labels

    return tokenized_inputs

In [42]:
# ============================================================
# 12. Tokenizar dataset
# ============================================================

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

print(tokenized_dataset)

Map:   0%|          | 0/34182 [00:00<?, ? examples/s]

Map:   0%|          | 0/4502 [00:00<?, ? examples/s]

Map:   0%|          | 0/4477 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 34182
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4502
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4477
    })
})


In [43]:
# ============================================================
# 13. Revisar ejemplo tokenizado
# ============================================================

example = tokenized_dataset["train"][0]

input_ids = example["input_ids"]
labels = example["labels"]

tokens = tokenizer.convert_ids_to_tokens(input_ids)

debug_rows = []

for token, label_id in zip(tokens, labels):
    if label_id == -100:
        label_name = "IGNORED"
    else:
        label_name = id2label[int(label_id)]

    debug_rows.append({
        "token": token,
        "label_id": label_id,
        "label": label_name,
    })

debug_df = pd.DataFrame(debug_rows)

display(debug_df.head(120))

,token,label_id,label
0,[CLS],-100,IGNORED
1,came,0,O
2,here,0,O
3,on,0,O
4,a,0,O
...,...,...,...
115,was,0,O
116,happy,0,O
117,to,0,O
118,bring,0,O


In [44]:
# ============================================================
# 14. QA de tokenización
# ============================================================

def tokenization_qa(tokenized_dataset):
    summary = {}

    for split_name in ["train", "validation", "test"]:
        split_data = tokenized_dataset[split_name]

        total_examples = len(split_data)
        total_tokens = 0
        supervised_tokens = 0
        ignored_tokens = 0
        label_counter = Counter()

        for example in split_data:
            labels = example["labels"]

            total_tokens += len(labels)

            for label_id in labels:
                if label_id == -100:
                    ignored_tokens += 1
                else:
                    supervised_tokens += 1
                    label_counter[id2label[int(label_id)]] += 1

        summary[split_name] = {
            "examples": int(total_examples),
            "total_subtokens": int(total_tokens),
            "supervised_tokens": int(supervised_tokens),
            "ignored_tokens": int(ignored_tokens),
            "label_counts": {
                str(k): int(v)
                for k, v in label_counter.items()
            },
            "label_percentages_supervised": {
                str(k): float(v / supervised_tokens * 100) if supervised_tokens else 0.0
                for k, v in label_counter.items()
            },
        }

    return summary


tokenization_summary = tokenization_qa(tokenized_dataset)

print(json.dumps(tokenization_summary, indent=2, ensure_ascii=False))

{
  "train": {
    "examples": 34182,
    "total_subtokens": 4416992,
    "supervised_tokens": 3929606,
    "ignored_tokens": 487386,
    "label_counts": {
      "O": 3836691,
      "B-DISH": 72137,
      "I-DISH": 20778
    },
    "label_percentages_supervised": {
      "O": 97.63551358584041,
      "B-DISH": 1.8357311140099035,
      "I-DISH": 0.5287553001496842
    }
  },
  "validation": {
    "examples": 4502,
    "total_subtokens": 577148,
    "supervised_tokens": 513682,
    "ignored_tokens": 63466,
    "label_counts": {
      "O": 502011,
      "B-DISH": 9067,
      "I-DISH": 2604
    },
    "label_percentages_supervised": {
      "O": 97.72797178020643,
      "B-DISH": 1.7650998088311447,
      "I-DISH": 0.5069284109624242
    }
  },
  "test": {
    "examples": 4477,
    "total_subtokens": 568217,
    "supervised_tokens": 506306,
    "ignored_tokens": 61911,
    "label_counts": {
      "O": 494898,
      "B-DISH": 8907,
      "I-DISH": 2501
    },
    "label_percentages_supervi

## 2. Entrenamiento del modelo NER

En este bloque se entrena un modelo transformer para token classification.

El modelo aprenderá a clasificar cada token como:

- `O`
- `B-DISH`
- `I-DISH`

Debido al fuerte desbalance de etiquetas, se utilizará una pérdida ponderada para dar más peso a las clases `B-DISH` e `I-DISH`.

La métrica principal será Entity-Level F1 mediante `seqeval`, ya que en NER no basta con medir accuracy token a token.

In [45]:
# ============================================================
# 15. Métricas NER con seqeval
# ============================================================

from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report as seqeval_classification_report
)

def align_predictions_for_metrics(predictions, labels):
    """
    Convierte logits y labels alineados en listas de etiquetas string,
    ignorando posiciones con -100.
    """

    preds = np.argmax(predictions, axis=2)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(preds, labels):
        current_preds = []
        current_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue

            current_preds.append(id2label[int(pred_id)])
            current_labels.append(id2label[int(label_id)])

        true_predictions.append(current_preds)
        true_labels.append(current_labels)

    return true_predictions, true_labels


def compute_metrics(eval_pred):
    """
    Métricas principales para NER.
    """

    predictions, labels = eval_pred

    true_predictions, true_labels = align_predictions_for_metrics(
        predictions,
        labels
    )

    entity_precision = precision_score(true_labels, true_predictions)
    entity_recall = recall_score(true_labels, true_predictions)
    entity_f1 = f1_score(true_labels, true_predictions)
    entity_accuracy = accuracy_score(true_labels, true_predictions)

    # Métricas token-level excluyendo posiciones -100.
    flat_true = []
    flat_pred = []

    preds = np.argmax(predictions, axis=2)

    for pred_seq, label_seq in zip(preds, labels):
        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue

            flat_true.append(int(label_id))
            flat_pred.append(int(pred_id))

    token_macro_f1 = 0.0
    token_weighted_f1 = 0.0

    if flat_true:
        from sklearn.metrics import f1_score as sklearn_f1_score

        token_macro_f1 = sklearn_f1_score(
            flat_true,
            flat_pred,
            average="macro",
            labels=[0, 1, 2],
            zero_division=0
        )

        token_weighted_f1 = sklearn_f1_score(
            flat_true,
            flat_pred,
            average="weighted",
            labels=[0, 1, 2],
            zero_division=0
        )

    return {
        "entity_precision": entity_precision,
        "entity_recall": entity_recall,
        "entity_f1": entity_f1,
        "entity_accuracy": entity_accuracy,
        "token_macro_f1": token_macro_f1,
        "token_weighted_f1": token_weighted_f1,
    }

In [46]:
# ============================================================
# 16. Cargar modelo Token Classification
# ============================================================

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(NER_LABELS),
    id2label=id2label,
    label2id=label2id,
)

print("Modelo cargado correctamente:")
print(MODEL_CHECKPOINT)
print("Número de etiquetas:", len(NER_LABELS))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Modelo cargado correctamente:
distilbert-base-uncased
Número de etiquetas: 3


In [47]:
# ============================================================
# 17. Calcular pesos de clase para pérdida ponderada
# ============================================================

def compute_label_counts_from_tokenized_split(tokenized_split):
    counter = Counter()

    for example in tqdm(tokenized_split, desc="Contando etiquetas supervisadas"):
        for label_id in example["labels"]:
            if label_id == -100:
                continue

            counter[int(label_id)] += 1

    return counter


train_label_counter = compute_label_counts_from_tokenized_split(
    tokenized_dataset["train"]
)

print("Conteo de etiquetas train:")
for label_id, count in sorted(train_label_counter.items()):
    print(id2label[label_id], "->", count)

total_supervised = sum(train_label_counter.values())
num_labels = len(NER_LABELS)

# Pesos balanceados clásicos: total / (num_labels * count)
# Como en NER el desbalance es muy fuerte, aplicamos raíz cuadrada
# para evitar pesos excesivamente agresivos.
raw_class_weights = []

for label_id in range(num_labels):
    count = train_label_counter.get(label_id, 1)
    weight = total_supervised / (num_labels * count)
    weight = np.sqrt(weight)
    raw_class_weights.append(weight)

# Cap para evitar inestabilidad si una clase tiene muy pocos ejemplos.
MAX_CLASS_WEIGHT = 8.0

class_weights = [
    min(float(weight), MAX_CLASS_WEIGHT)
    for weight in raw_class_weights
]

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float
)

print("\nPesos de clase usados:")
for label_id, weight in enumerate(class_weights):
    print(id2label[label_id], "->", round(weight, 4))

Contando etiquetas supervisadas:   0%|          | 0/34182 [00:00<?, ?it/s]

Conteo de etiquetas train:
O -> 3836691
B-DISH -> 72137
I-DISH -> 20778

Pesos de clase usados:
O -> 0.5843
B-DISH -> 4.2612
I-DISH -> 7.9398


In [48]:
# ============================================================
# 18. Trainer con pérdida ponderada - versión robusta DataParallel
# ============================================================

USE_CLASS_WEIGHTS = True

class WeightedTokenClassificationTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")

        outputs = model(**inputs)
        logits = outputs.logits

        if self.class_weights is not None:
            loss_fct = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device),
                ignore_index=-100
            )
        else:
            loss_fct = torch.nn.CrossEntropyLoss(
                ignore_index=-100
            )

        # Importante:
        # En Kaggle, si hay varias GPU, el modelo puede estar envuelto en DataParallel.
        # Por eso evitamos model.config.num_labels y usamos directamente logits.shape[-1].
        num_labels = logits.shape[-1]

        loss = loss_fct(
            logits.view(-1, num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

In [49]:
# ============================================================
# 19. TrainingArguments robusto
# ============================================================

if USE_SMOKE_TEST:
    NUM_EPOCHS = 3
    TRAIN_BATCH_SIZE = 16
    EVAL_BATCH_SIZE = 32
    LOGGING_STEPS = 25
else:
    NUM_EPOCHS = 2
    TRAIN_BATCH_SIZE = 16
    EVAL_BATCH_SIZE = 32
    LOGGING_STEPS = 200

training_output_dir = MODEL_DIR / (
    "checkpoints_smoke" if USE_SMOKE_TEST else "checkpoints_full"
)

def build_training_args():
    base_kwargs = dict(
        output_dir=str(training_output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=LOGGING_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="entity_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to=[],
        fp16=torch.cuda.is_available(),
        seed=RANDOM_STATE,
    )

    try:
        return TrainingArguments(
            eval_strategy="epoch",
            **base_kwargs
        )
    except TypeError:
        return TrainingArguments(
            evaluation_strategy="epoch",
            **base_kwargs
        )

training_args = build_training_args()

print(training_args)

TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_strategy=IntervalStrategy.EPOCH,
eval_use_gather_object=Fals

In [50]:
# ============================================================
# 20. Crear Trainer
# ============================================================

common_trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

def create_trainer():
    trainer_class = WeightedTokenClassificationTrainer if USE_CLASS_WEIGHTS else Trainer

    extra_kwargs = {}

    if USE_CLASS_WEIGHTS:
        extra_kwargs["class_weights"] = class_weights_tensor

    # Compatibilidad con versiones nuevas/antiguas de transformers.
    try:
        return trainer_class(
            **common_trainer_kwargs,
            processing_class=tokenizer,
            **extra_kwargs
        )
    except TypeError as e:
        if "processing_class" in str(e) or "unexpected keyword" in str(e):
            print("processing_class no aceptado. Creando Trainer sin processing_class.")
            return trainer_class(
                **common_trainer_kwargs,
                **extra_kwargs
            )

        raise e


trainer = create_trainer()

print("Trainer creado correctamente.")

Trainer creado correctamente.


In [51]:
# ============================================================
# 21. Entrenar modelo
# ============================================================

train_result = trainer.train()

print("Entrenamiento finalizado.")
print(train_result)

Epoch,Training Loss,Validation Loss,Entity Precision,Entity Recall,Entity F1,Entity Accuracy,Token Macro F1,Token Weighted F1
1,0.026985,0.026853,0.831693,0.941215,0.883071,0.993229,0.867579,0.994043
2,0.015154,0.018409,0.909515,0.966692,0.937233,0.996511,0.920126,0.996753


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Entrenamiento finalizado.
TrainOutput(global_step=2138, training_loss=0.04208457101941443, metrics={'train_runtime': 972.6476, 'train_samples_per_second': 70.287, 'train_steps_per_second': 2.198, 'total_flos': 4465109087315100.0, 'train_loss': 0.04208457101941443, 'epoch': 2.0})


In [52]:
# ============================================================
# 22. Evaluar validation y test
# ============================================================

validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
test_metrics = trainer.evaluate(tokenized_dataset["test"])

print("Validation metrics:")
print(json.dumps(validation_metrics, indent=2))

print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2))

Validation metrics:
{
  "eval_loss": 0.018409064039587975,
  "eval_entity_precision": 0.9095154093597593,
  "eval_entity_recall": 0.9666924010146686,
  "eval_entity_f1": 0.9372326775021386,
  "eval_entity_accuracy": 0.9965114603976779,
  "eval_token_macro_f1": 0.9201262235645206,
  "eval_token_weighted_f1": 0.9967526494152149,
  "eval_runtime": 24.0584,
  "eval_samples_per_second": 187.128,
  "eval_steps_per_second": 2.951,
  "epoch": 2.0
}

Test metrics:
{
  "eval_loss": 0.017678597941994667,
  "eval_entity_precision": 0.9093591905564924,
  "eval_entity_recall": 0.9686763219939374,
  "eval_entity_f1": 0.938081000271813,
  "eval_entity_accuracy": 0.9965850691084048,
  "eval_token_macro_f1": 0.9211308110739854,
  "eval_token_weighted_f1": 0.9968114831169683,
  "eval_runtime": 24.0602,
  "eval_samples_per_second": 186.075,
  "eval_steps_per_second": 2.909,
  "epoch": 2.0
}


In [53]:
# ============================================================
# 23. Predicciones test y report seqeval
# ============================================================

test_predictions_output = trainer.predict(tokenized_dataset["test"])

test_logits = test_predictions_output.predictions
test_labels = test_predictions_output.label_ids

test_true_predictions, test_true_labels = align_predictions_for_metrics(
    test_logits,
    test_labels
)

print("Seqeval classification report:")
print(
    seqeval_classification_report(
        test_true_labels,
        test_true_predictions,
        digits=4,
        zero_division=0
    )
)

Seqeval classification report:
              precision    recall  f1-score   support

        DISH     0.9094    0.9687    0.9381      8907

   micro avg     0.9094    0.9687    0.9381      8907
   macro avg     0.9094    0.9687    0.9381      8907
weighted avg     0.9094    0.9687    0.9381      8907



In [54]:
# ============================================================
# 24. Matriz de confusión token-level
# ============================================================

flat_true_labels = []
flat_pred_labels = []

test_pred_ids = np.argmax(test_logits, axis=2)

for pred_seq, label_seq in zip(test_pred_ids, test_labels):
    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id == -100:
            continue

        flat_true_labels.append(id2label[int(label_id)])
        flat_pred_labels.append(id2label[int(pred_id)])

token_cm = confusion_matrix(
    flat_true_labels,
    flat_pred_labels,
    labels=NER_LABELS
)

token_cm_df = pd.DataFrame(
    token_cm,
    index=[f"true_{label}" for label in NER_LABELS],
    columns=[f"pred_{label}" for label in NER_LABELS]
)

display(token_cm_df)

print("\nToken-level classification report:")
print(
    classification_report(
        flat_true_labels,
        flat_pred_labels,
        labels=NER_LABELS,
        digits=4,
        zero_division=0
    )
)

,pred_O,pred_B-DISH,pred_I-DISH
true_O,493365,474,1059
true_B-DISH,34,8775,98
true_I-DISH,40,24,2437



Token-level classification report:
              precision    recall  f1-score   support

           O     0.9999    0.9969    0.9984    494898
      B-DISH     0.9463    0.9852    0.9653      8907
      I-DISH     0.6781    0.9744    0.7997      2501

    accuracy                         0.9966    506306
   macro avg     0.8747    0.9855    0.9211    506306
weighted avg     0.9973    0.9966    0.9968    506306



In [55]:
# ============================================================
# 25. Guardar métricas
# ============================================================

run_name = "smoke" if USE_SMOKE_TEST else "full"

metrics_output = {
    "notebook": "06_dish_ner_transformer_training",
    "run_name": run_name,
    "model_checkpoint": MODEL_CHECKPOINT,
    "dataset_path": str(BIO_DATASET_PATH),
    "max_length": MAX_LENGTH,
    "num_epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "label_all_tokens": LABEL_ALL_TOKENS,
    "use_class_weights": USE_CLASS_WEIGHTS,
    "class_weights": {
        id2label[i]: float(class_weights_tensor[i].item())
        for i in range(len(NER_LABELS))
    },
    "labels": NER_LABELS,
    "label2id": label2id,
    "id2label": {
        str(k): v
        for k, v in id2label.items()
    },
    "validation_metrics": {
        str(k): float(v)
        for k, v in validation_metrics.items()
        if isinstance(v, (int, float, np.integer, np.floating))
    },
    "test_metrics": {
        str(k): float(v)
        for k, v in test_metrics.items()
        if isinstance(v, (int, float, np.integer, np.floating))
    },
    "token_confusion_matrix": token_cm_df.to_dict(),
}

metrics_path = METRICS_DIR / f"dish_ner_transformer_metrics_{run_name}.json"

with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics_output, f, indent=2, ensure_ascii=False)

print("Métricas guardadas en:")
print(metrics_path)

Métricas guardadas en:
/kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer/metrics/dish_ner_transformer_metrics_full.json


In [56]:
# ============================================================
# 26. Guardar modelo entrenado
# ============================================================

final_model_path = MODEL_DIR / (
    "dish_ner_transformer_smoke" if USE_SMOKE_TEST else "dish_ner_transformer_full"
)

trainer.save_model(str(final_model_path))
tokenizer.save_pretrained(str(final_model_path))

print("Modelo guardado en:")
print(final_model_path)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Modelo guardado en:
/kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer/model/dish_ner_transformer_full


## 3. Inferencia y revisión de predicciones

Tras entrenar el modelo NER, se realiza una fase de inferencia y análisis cualitativo.

El objetivo es comprobar:

- qué entidades predice el modelo en textos nuevos;
- cómo se comporta en ejemplos mixtos;
- si tiende a sobreextender entidades;
- qué errores aparecen en test;
- si el modelo es utilizable como primer detector de platos para Hidden Gems.

Aunque las métricas son altas, este análisis es necesario porque el dataset procede de weak labeling y no de anotación humana completa.

In [57]:
# ============================================================
# 27. Funciones BIO para extracción de entidades
# ============================================================

def extract_entities_from_bio(tokens, labels, entity_label="DISH"):
    """
    Extrae entidades desde tokens + etiquetas BIO.
    Devuelve spans a nivel de índice de token.
    """

    entities = []
    current_entity = None

    for idx, (token, label) in enumerate(zip(tokens, labels)):
        if label == f"B-{entity_label}":
            if current_entity is not None:
                entities.append(current_entity)

            current_entity = {
                "label": entity_label,
                "start_token": idx,
                "end_token": idx + 1,
                "tokens": [token],
            }

        elif label == f"I-{entity_label}":
            if current_entity is not None:
                current_entity["end_token"] = idx + 1
                current_entity["tokens"].append(token)
            else:
                # Caso defensivo: I sin B previa
                current_entity = {
                    "label": entity_label,
                    "start_token": idx,
                    "end_token": idx + 1,
                    "tokens": [token],
                }

        else:
            if current_entity is not None:
                entities.append(current_entity)
                current_entity = None

    if current_entity is not None:
        entities.append(current_entity)

    for ent in entities:
        ent["text"] = " ".join(ent["tokens"])
        ent["normalized_text"] = ent["text"].lower()

    return entities


def render_tokens_with_entities(tokens, labels):
    """
    Renderiza tokens con entidades marcadas.
    """

    output = []
    inside = False

    for token, label in zip(tokens, labels):
        if label == "B-DISH":
            if inside:
                output.append("]")
            output.append("[DISH:")
            output.append(token)
            inside = True

        elif label == "I-DISH":
            if inside:
                output.append(" " + token)
            else:
                output.append("[DISH:" + token)
                inside = True

        else:
            if inside:
                output.append("]")
                inside = False
            output.append(" " + token)

    if inside:
        output.append("]")

    return "".join(output).strip()

In [58]:
# ============================================================
# 28. Construir predicciones legibles sobre test
# ============================================================

# Reutiliza test_predictions_output si ya existe.
# Si no existe, lo generamos.
if "test_predictions_output" not in globals():
    test_predictions_output = trainer.predict(tokenized_dataset["test"])

test_logits = test_predictions_output.predictions
test_labels = test_predictions_output.label_ids
test_pred_ids = np.argmax(test_logits, axis=2)

test_records = []

for i in tqdm(range(len(test_pd)), desc="Construyendo predicciones legibles"):
    original_tokens = test_pd.iloc[i]["tokens"]

    pred_seq = test_pred_ids[i]
    label_seq = test_labels[i]

    word_true_labels = []
    word_pred_labels = []

    for pred_id, label_id in zip(pred_seq, label_seq):
        if label_id == -100:
            continue

        word_true_labels.append(id2label[int(label_id)])
        word_pred_labels.append(id2label[int(pred_id)])

    # Por truncation, puede haber menos tokens supervisados que tokens originales.
    supervised_token_count = len(word_true_labels)
    visible_tokens = original_tokens[:supervised_token_count]

    true_entities = extract_entities_from_bio(visible_tokens, word_true_labels)
    pred_entities = extract_entities_from_bio(visible_tokens, word_pred_labels)

    test_records.append({
        "review_id": test_pd.iloc[i]["review_id"],
        "split": test_pd.iloc[i]["split"],
        "entity_count_gold": int(test_pd.iloc[i]["entity_count"]),
        "supervised_token_count": int(supervised_token_count),
        "tokens": visible_tokens,
        "true_labels": word_true_labels,
        "pred_labels": word_pred_labels,
        "true_entities": true_entities,
        "pred_entities": pred_entities,
        "true_entity_count_visible": len(true_entities),
        "pred_entity_count": len(pred_entities),
        "render_true": render_tokens_with_entities(visible_tokens, word_true_labels),
        "render_pred": render_tokens_with_entities(visible_tokens, word_pred_labels),
    })

test_prediction_readable_df = pd.DataFrame(test_records)

display(test_prediction_readable_df.head(3))

Construyendo predicciones legibles:   0%|          | 0/4477 [00:00<?, ?it/s]

,review_id,split,entity_count_gold,supervised_token_count,tokens,true_labels,pred_labels,true_entities,pred_entities,true_entity_count_visible,pred_entity_count,render_true,render_pred
0,RAhYHS028TKwAai7eFM4rA,test,2,50,"[Since, going, paleo, it's, hard, to, find, something, that, doesn't, taste, like, cardboard, to, satisfy, my, pizza, cravings, ., The, gluten, free, pizza, here, is, the, best, I've, found, and, they, even, personalize, it, any, way, I, want, including, no, cheese, and, extra, sauce, ., It's, m...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[{'label': 'DISH', 'start_token': 16, 'end_token': 17, 'tokens': ['pizza'], 'text': 'pizza', 'normalized_text': 'pizza'}, {'label': 'DISH', 'start_token': 22, 'end_token': 23, 'tokens': ['pizza'], 'text': 'pizza', 'normalized_text': 'pizza'}]","[{'label': 'DISH', 'start_token': 16, 'end_token': 17, 'tokens': ['pizza'], 'text': 'pizza', 'normalized_text': 'pizza'}, {'label': 'DISH', 'start_token': 22, 'end_token': 23, 'tokens': ['pizza'], 'text': 'pizza', 'normalized_text': 'pizza'}]",2,2,Since going paleo it's hard to find something that doesn't taste like cardboard to satisfy my[DISH:pizza] cravings . The gluten free[DISH:pizza] here is the best I've found and they even personalize it any way I want including no cheese and extra sauce . It's my new fav !,Since going paleo it's hard to find something that doesn't taste like cardboard to satisfy my[DISH:pizza] cravings . The gluten free[DISH:pizza] here is the best I've found and they even personalize it any way I want including no cheese and extra sauce . It's my new fav !
1,pKcQBSuGskizQ992yCRJTg,test,2,66,"[Wasn't, too, impressed, with, their, food, ., Ordered, the, tuna, tartare, ,, which, never, came, out, :, (, ., Had, the, sirloin, steak, ,, which, wasn't, even, that, good, ,, and, it, was, also, overcooked, ., Fortunately, I, was, with, good, company, ,, so, the, food, didn't, bother, me, as,...","[O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, B-DISH, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O]","[{'label': 'DISH', 'start_token': 9, 'end_token': 10, 'tokens': ['tuna'], 'text': 'tuna', 'normalized_text': 'tuna'}, {'label': 'DISH', 'start_token': 22, 'end_token': 23, 'tokens': ['steak'], 'text': 'steak', 'normalized_text': 'steak'}]","[{'label': 'DISH', 'start_token': 9, 'end_token': 10, 'tokens': ['tuna'], 'text': 'tuna', 'normalized_text': 'tuna'}, {'label': 'DISH', 'start_token': 22, 'end_token': 23, 'tokens': ['steak'], 'text': 'steak', 'normalized_text': 'steak'}]",2,2,"Wasn't too impressed with their food . Ordered the[DISH:tuna] tartare , which never came out : ( . Had the sirloin[DISH:steak] , which wasn't even that good , and it was also overcooked . Fortunately I was with good company , so the food didn't bother me as much . However , I probably won't come...","Wasn't too impressed with their food . Ordered the[DISH:tuna] tartare , which never came out : ( . Had the sirloin[DISH:steak] , which wasn't even that good , and it was also overcooked . Fortunately I was with good company , so the food didn't bother me as much . However , I probably won't come..."
2,dKdDqZAKjtXXoRRpMtnhSA,test,13,231,"[I, had, very, mixed, feelings, about, Baileys, ', Range, ., I, felt, like, this, restaurant, did, many, things, right, ,, and, a, couple, of, things, wrong, ., I'll, start, out, with, the, good, ., The, atmosphere, and, serv

In [59]:
# ============================================================
# 29. Análisis simple de errores por entidades
# ============================================================

def entity_key(entity):
    return (
        int(entity["start_token"]),
        int(entity["end_token"]),
        entity["normalized_text"]
    )


error_rows = []

for row in test_prediction_readable_df.itertuples(index=False):
    true_set = set(entity_key(ent) for ent in row.true_entities)
    pred_set = set(entity_key(ent) for ent in row.pred_entities)

    tp = len(true_set & pred_set)
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)

    error_rows.append({
        "review_id": row.review_id,
        "true_entity_count": len(true_set),
        "pred_entity_count": len(pred_set),
        "tp_entities": tp,
        "fp_entities": fp,
        "fn_entities": fn,
        "has_entity_error": fp > 0 or fn > 0,
        "render_true": row.render_true,
        "render_pred": row.render_pred,
    })

entity_error_df = pd.DataFrame(error_rows)

print("Total test docs:", len(entity_error_df))
print("Docs con error de entidad:", int(entity_error_df["has_entity_error"].sum()))
print("Porcentaje docs con error:", round(entity_error_df["has_entity_error"].mean() * 100, 2), "%")

print("\nResumen TP/FP/FN:")
display(
    entity_error_df[["tp_entities", "fp_entities", "fn_entities"]]
    .sum()
    .to_frame("count")
)

print("\nDocs con más falsos positivos:")
display(
    entity_error_df
    .sort_values(["fp_entities", "fn_entities"], ascending=False)
    .head(10)
    [["review_id", "true_entity_count", "pred_entity_count", "tp_entities", "fp_entities", "fn_entities", "render_true", "render_pred"]]
)

print("\nDocs con más falsos negativos:")
display(
    entity_error_df
    .sort_values(["fn_entities", "fp_entities"], ascending=False)
    .head(10)
    [["review_id", "true_entity_count", "pred_entity_count", "tp_entities", "fp_entities", "fn_entities", "render_true", "render_pred"]]
)

Total test docs: 4477
Docs con error de entidad: 666
Porcentaje docs con error: 14.88 %

Resumen TP/FP/FN:


,count
tp_entities,8628
fp_entities,860
fn_entities,279



Docs con más falsos positivos:


,review_id,true_entity_count,pred_entity_count,tp_entities,fp_entities,fn_entities,render_true,render_pred
315,bqYUF7TKmJARZS0x2SsuxA,2,11,2,9,0,"I had the[DISH:beef noodle soup lunch special] and a taro milk tea with boba . Starting off the review with the boba . The taro milk tea tasted really good , but unfortunately the boba itself was bad . The boba was hard and seemed undercooked . My suggestion is to get the milk tea but hold the b...","I had the[DISH:beef noodle soup lunch special] and a[DISH:taro] milk[DISH:tea] with boba . Starting off the review with the boba . The[DISH:taro milk tea tasted really good] , but unfortunately the boba itself was bad . The boba was hard and seemed undercooked . My suggestion is to get the milk[..."
1331,gzb88mpKPMnwOhYhYFVY_A,4,8,3,5,1,"I ate here with my husband for dinner and was pleasantly surprised by the friendly staff , prompt service , and excellent food . Upon walking in , we were greeted by a hostess as well as the manager who both were very kind . Our server was excellent with helping us choose off of the menu , refil...","I ate here with my husband for dinner and was pleasantly surprised by the friendly staff , prompt service , and excellent food . Upon walking in , we were greeted by a hostess as well as the manager who both were very kind . Our server was excellent with helping us choose off of the menu , refil..."
806,0Q4velGF_UzuOVRTzSWp4A,2,7,2,5,0,"Everything is covered in pink , including the glittery baked goods . And everything is a bit pricey . It's really a treat yourself type of splurge . It gets crowded and the tables are tiny , so plan accordingly . As for the food : Cakes - heavy on buttercream frosting , so your experience partly...","Everything is covered in pink , including the glittery baked goods . And everything is a bit pricey . It's really a treat yourself type of splurge . It gets crowded and the tables are tiny , so plan accordingly . As for the food : Cakes - heavy on buttercream frosting , so your experience partly..."
848,Rpf8uSiaMhxXGSqF6uMFLw,1,6,1,5,0,"Atmosphere - 5 Stars Very clean environment with simple , yet elegant historic pieces . Very laid back staff . Food - 5 Stars I had the[DISH:Steak] and Egg Sandwich . It was an interesting list of ingredients that all complimented each other well . Arugula folded eggs with jalapeño jam , shaved ...","Atmosphere - 5 Stars Very clean environment with simple , yet elegant historic pieces . Very laid back staff . Food - 5 Stars I had the[DISH:Steak] and Egg Sandwich . It was an interesting list of ingredients that all complimented each other well . Arugula folded eggs with jalapeño jam , shaved ..."
280,oWJRDDsiE77bRcAh3uj9Lg,2,5,1,4,1,"I stumbled upon this place thinking it was just Chinese food , but it's actually Malaysian . It was actually a pleasant surprise because that's one of my new favorite types of cuisine ! I devoured the Roti Canai in record time . The people around me looked a little scared , but I couldn't resist...","I stumbled upon this place thinking it was just Chinese food , but it's actually Malaysian . It was actually a pleasant surprise because that's one of my new favorite types of cuisine ! I devoured the Roti Canai in record time . The people around me looked a little scared , but I couldn't resist..."
2620,KLpWabgPdgsTRPyG_HIS0A,4,7,3,4,1,"I'd never heard of Zoup before , but apparently it's an east coast chain restaurant . Don't let that scare you away , though . The soups are . . . well . . . souper ! : ) There were about a dozen different soup selections and they freely provided as many samples as you wished . Oh , and they do ...","I'd never heard of Zoup before , but apparently it's an east coast chain restaurant . Don't let that scare you away , though . The soups are . . . well . . . souper ! : ) There were about a dozen different soup selections and they freely provided as many samples as you wished . Oh , and they do ..."
1513,P5NkbE5xKPEVkXNr9CHQFg,2,6,2,4,0,"I fin


Docs con más falsos negativos:


,review_id,true_entity_count,pred_entity_count,tp_entities,fp_entities,fn_entities,render_true,render_pred
1874,edKBLpKMu4fsAgQfFpC4Bw,3,3,0,3,3,"I am not a huge fan of[DISH:Burger] King . I'm really not . But sometimes , when you have kids , grandkids , a full time job , school and 25 hours worth of things to do in a 24 hour day , sometimes , it's a necessary evil . I've never been in this particular[DISH:Burger] King before , however , ...","I am not a huge fan of[DISH:Burger King] . I'm really not . But sometimes , when you have kids , grandkids , a full time job , school and 25 hours worth of things to do in a 24 hour day , sometimes , it's a necessary evil . I've never been in this particular[DISH:Burger King] before , however , ..."
3238,b2UlfmPr5uFbfFtwSB8-Pw,5,4,2,2,3,Let me tell you about this Red[DISH:Lobster] . . . they do a great job ! I ordered the[DISH:lobster] Maine with[DISH:steak] - this option gives you a choice of the filet mignon or Strip . I went with the strip . This meal comes with a salad . I ordered the caesar . . . and Mojitos . My[DISH:stea...,Let me tell you about this Red[DISH:Lobster] . . . they do a great job ! I ordered the[DISH:lobster Maine with steak] - this option gives you a choice of the filet mignon or Strip . I went with the strip . This meal comes with a salad . I ordered the caesar . . . and Mojitos . My[DISH:steak] ( m...
69,4JLyCSUrPig82bCMcXAFlQ,6,7,4,3,2,"I'm very conflicted about this place . The food was mostly excellent . The service was mostly atrocious . When we arrived , the server took our order , but did not write anything down ( this will come back later ) , and brought out our drinks . The appetizer was correct , and was delicious ; we ...","I'm very conflicted about this place . The food was mostly excellent . The service was mostly atrocious . When we arrived , the server took our order , but did not write anything down ( this will come back later ) , and brought out our drinks . The appetizer was correct , and was delicious ; we ..."
111,tR3IWkyDhF3uohqBGc0GaQ,2,3,0,3,2,Went to Khyber Pass Pub last evening with some friends and have to say I was a little disappointed . After reading so many awesome Yelp reviews we were very excited to try the place out . Dinner was at 8 : 00 and we were lucky to get seated right away . The place was really loud so it was hard t...,Went to Khyber Pass Pub last evening with some friends and have to say I was a little disappointed . After reading so many awesome Yelp reviews we were very excited to try the place out . Dinner was at 8 : 00 and we were lucky to get seated right away . The place was really loud so it was hard t...
2210,IPiSspbDxPsH5e5jrW22EA,2,3,0,3,2,"Ordered from Pizzarella a few times . My favorite thing is the[DISH:Penne] Pasta with[DISH:Eggplant] on top . his is not a menu item , but something they are willing to do , and it's delicious and a HUGE portion ! The staff has always been pleasant on the phone , the delivery guy Scott is a gem ...","Ordered from[DISH:Pizzarella] a few times . My favorite thing is the[DISH:Penne Pasta with][DISH:Eggplant on top] . his is not a menu item , but something they are willing to do , and it's delicious and a HUGE portion ! The staff has always been pleasant on the phone , the delivery guy Scott is ..."
3396,aRWYMRTsz0bwd5J0_zzwhA,7,8,5,3,2,"First of all the interior is nice , really intimate . If you get a chance check out the upstairs lounge that overlooks the street ! SO , to the food . . . We started with the Poke . I thought it was pretty good , a lot of interesting textures and bites but a little to acidic . The hazelnut and r...","First of all the interior is nice , really intimate . If you get a chance check out the upstairs lounge that overlooks the street ! SO , to the food . . . We started with the Poke . I thought it was pretty good , a lot of interesting textures and bites but a little to acidic . The hazelnut and r..."
3899,Ow1W876MDd3at23PzBGqEg,5,6,3,3,2,"This restaura

In [61]:
# ============================================================
# 30. Función de inferencia sobre texto nuevo
# ============================================================
import re

SIMPLE_TOKEN_PATTERN = re.compile(
    r"\w+(?:[-']\w+)*|[^\w\s]",
    flags=re.UNICODE
)

def simple_tokenize(text):
    return [match.group(0) for match in SIMPLE_TOKEN_PATTERN.finditer(text)]


def predict_dishes_in_text(text: str):
    """
    Predice menciones DISH en un texto nuevo.
    """

    tokens = simple_tokenize(text)

    if not tokens:
        return {
            "text": text,
            "tokens": [],
            "pred_labels": [],
            "entities": [],
            "rendered": "",
        }

    encoding = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    model_device = next(trainer.model.parameters()).device
    encoding = {k: v.to(model_device) for k, v in encoding.items()}

    trainer.model.eval()

    with torch.no_grad():
        outputs = trainer.model(**encoding)
        logits = outputs.logits
        pred_ids = torch.argmax(logits, dim=-1).cpu().numpy()[0]

    # Recuperar word_ids desde una segunda tokenización sin tensores
    encoding_for_word_ids = tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=MAX_LENGTH
    )

    word_ids = encoding_for_word_ids.word_ids()

    word_predictions = {}
    previous_word_id = None

    for pred_id, word_id in zip(pred_ids, word_ids):
        if word_id is None:
            continue

        # Solo primer subtoken, coherente con LABEL_ALL_TOKENS=False
        if word_id != previous_word_id:
            word_predictions[word_id] = id2label[int(pred_id)]

        previous_word_id = word_id

    supervised_token_count = max(word_predictions.keys()) + 1 if word_predictions else 0
    visible_tokens = tokens[:supervised_token_count]

    pred_labels = [
        word_predictions.get(i, "O")
        for i in range(supervised_token_count)
    ]

    entities = extract_entities_from_bio(visible_tokens, pred_labels)

    return {
        "text": text,
        "tokens": visible_tokens,
        "pred_labels": pred_labels,
        "entities": entities,
        "rendered": render_tokens_with_entities(visible_tokens, pred_labels),
    }

In [62]:
# ============================================================
# 31. Probar inferencia manual
# ============================================================

manual_texts = [
    "The crab legs were amazing, but the fries were cold.",
    "I ordered the burger with bacon and the mac and cheese.",
    "The service was terrible, but the pasta was delicious.",
    "We loved the sushi, the salmon and the spicy tuna tacos.",
    "The atmosphere was nice, but the food was average.",
    "Their fried chicken and mashed potatoes were fantastic.",
    "I would come back just for the ice cream and the pumpkin pie.",
]

manual_predictions = []

for text in manual_texts:
    result = predict_dishes_in_text(text)

    manual_predictions.append({
        "text": text,
        "rendered_prediction": result["rendered"],
        "entities": result["entities"],
    })

manual_predictions_df = pd.DataFrame(manual_predictions)

display(manual_predictions_df)

,text,rendered_prediction,entities
0,"The crab legs were amazing, but the fries were cold.","The[DISH:crab legs] were amazing , but the[DISH:fries] were cold .","[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'tokens': ['crab', 'legs'], 'text': 'crab legs', 'normalized_text': 'crab legs'}, {'label': 'DISH', 'start_token': 8, 'end_token': 9, 'tokens': ['fries'], 'text': 'fries', 'normalized_text': 'fries'}]"
1,I ordered the burger with bacon and the mac and cheese.,I ordered the[DISH:burger with bacon] and the[DISH:mac and cheese] .,"[{'label': 'DISH', 'start_token': 3, 'end_token': 6, 'tokens': ['burger', 'with', 'bacon'], 'text': 'burger with bacon', 'normalized_text': 'burger with bacon'}, {'label': 'DISH', 'start_token': 8, 'end_token': 11, 'tokens': ['mac', 'and', 'cheese'], 'text': 'mac and cheese', 'normalized_text': ..."
2,"The service was terrible, but the pasta was delicious.","The service was terrible , but the[DISH:pasta] was delicious .","[{'label': 'DISH', 'start_token': 7, 'end_token': 8, 'tokens': ['pasta'], 'text': 'pasta', 'normalized_text': 'pasta'}]"
3,"We loved the sushi, the salmon and the spicy tuna tacos.","We loved the[DISH:sushi] , the[DISH:salmon] and the spicy[DISH:tuna][DISH:tacos] .","[{'label': 'DISH', 'start_token': 3, 'end_token': 4, 'tokens': ['sushi'], 'text': 'sushi', 'normalized_text': 'sushi'}, {'label': 'DISH', 'start_token': 6, 'end_token': 7, 'tokens': ['salmon'], 'text': 'salmon', 'normalized_text': 'salmon'}, {'label': 'DISH', 'start_token': 10, 'end_token': 11, ..."
4,"The atmosphere was nice, but the food was average.","The atmosphere was nice , but the food was average .",[]
5,Their fried chicken and mashed potatoes were fantastic.,Their[DISH:fried chicken] and[DISH:mashed potatoes] were fantastic .,"[{'label': 'DISH', 'start_token': 1, 'end_token': 3, 'tokens': ['fried', 'chicken'], 'text': 'fried chicken', 'normalized_text': 'fried chicken'}, {'label': 'DISH', 'start_token': 4, 'end_token': 6, 'tokens': ['mashed', 'potatoes'], 'text': 'mashed potatoes', 'normalized_text': 'mashed potatoes'}]"
6,I would come back just for the ice cream and the pumpkin pie.,I would come back just for the[DISH:ice cream] and the[DISH:pumpkin pie] .,"[{'label': 'DISH', 'start_token': 7, 'end_token': 9, 'tokens': ['ice', 'cream'], 'text': 'ice cream', 'normalized_text': 'ice cream'}, {'label': 'DISH', 'start_token': 11, 'end_token': 13, 'tokens': ['pumpkin', 'pie'], 'text': 'pumpkin pie', 'normalized_text': 'pumpkin pie'}]"


In [64]:
# ============================================================
# 31.5. Función auxiliar para guardar JSONL
# ============================================================

def make_json_safe(value):
    """
    Convierte tipos de numpy/pandas y objetos especiales a tipos serializables en JSON.
    """

    if isinstance(value, (np.integer,)):
        return int(value)

    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)

    if isinstance(value, (np.bool_,)):
        return bool(value)

    if isinstance(value, float) and np.isnan(value):
        return None

    if isinstance(value, set):
        return sorted(list(value))

    if isinstance(value, Path):
        return str(value)

    return value


def save_jsonl(df_to_save: pd.DataFrame, path: Path):
    """
    Guarda un DataFrame como JSONL de forma robusta.
    """

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as f:
        for record in df_to_save.to_dict(orient="records"):
            clean_record = {
                str(key): make_json_safe(value)
                for key, value in record.items()
            }

            f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

    print(f"Guardado: {path} ({len(df_to_save):,} registros)")

In [65]:
# ============================================================
# 32. Guardar predicciones y análisis
# ============================================================

run_name = "smoke" if USE_SMOKE_TEST else "full"

readable_predictions_path = PREDICTIONS_DIR / f"dish_ner_test_predictions_readable_{run_name}.jsonl"
entity_error_path = METRICS_DIR / f"dish_ner_entity_error_analysis_{run_name}.json"
manual_predictions_path = PREDICTIONS_DIR / f"dish_ner_manual_predictions_{run_name}.jsonl"

save_jsonl(test_prediction_readable_df, readable_predictions_path)
save_jsonl(manual_predictions_df, manual_predictions_path)

entity_error_summary = {
    "run_name": run_name,
    "total_test_docs": int(len(entity_error_df)),
    "docs_with_entity_error": int(entity_error_df["has_entity_error"].sum()),
    "docs_with_entity_error_percentage": float(entity_error_df["has_entity_error"].mean() * 100),
    "total_tp_entities": int(entity_error_df["tp_entities"].sum()),
    "total_fp_entities": int(entity_error_df["fp_entities"].sum()),
    "total_fn_entities": int(entity_error_df["fn_entities"].sum()),
    "notes": (
        "Entity comparison uses exact token span and normalized entity text. "
        "This is stricter than text-only comparison."
    )
}

with entity_error_path.open("w", encoding="utf-8") as f:
    json.dump(entity_error_summary, f, indent=2, ensure_ascii=False)

print("Predicciones test guardadas en:")
print(readable_predictions_path)

print("\nPredicciones manuales guardadas en:")
print(manual_predictions_path)

print("\nResumen de errores guardado en:")
print(entity_error_path)

print(json.dumps(entity_error_summary, indent=2, ensure_ascii=False))

Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer/predictions/dish_ner_test_predictions_readable_full.jsonl (4,477 registros)
Guardado: /kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer/predictions/dish_ner_manual_predictions_full.jsonl (7 registros)
Predicciones test guardadas en:
/kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer/predictions/dish_ner_test_predictions_readable_full.jsonl

Predicciones manuales guardadas en:
/kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer/predictions/dish_ner_manual_predictions_full.jsonl

Resumen de errores guardado en:
/kaggle/working/hidden_gems_ai/outputs/dish_ner_transformer/metrics/dish_ner_entity_error_analysis_full.json
{
  "run_name": "full",
  "total_test_docs": 4477,
  "docs_with_entity_error": 666,
  "docs_with_entity_error_percentage": 14.87603305785124,
  "total_tp_entities": 8628,
  "total_fp_entities": 860,
  "total_fn_entities": 279,
  "notes": "Entity comparison uses exact token span 

## Conclusión del Notebook 06

En este notebook se ha entrenado el primer modelo NER de Hidden Gems para detección de platos.

El modelo final `distilbert-base-uncased` entrenado sobre el dataset BIO high precision obtiene en test:

- Entity Precision: 0.9094
- Entity Recall: 0.9687
- Entity F1: 0.9381
- Token Macro F1: 0.9211
- Token Weighted F1: 0.9968

Estos resultados indican que el modelo aprende correctamente a detectar menciones de platos sobre el dataset weak-supervised generado previamente.

No obstante, el resultado debe interpretarse con cautela porque el dataset procede de weak labeling automático. Por tanto, el modelo `Dish NER v1` queda validado como primer detector automático, pero deberá evaluarse más adelante sobre una muestra revisada manualmente y sobre reviews reales en español de Google Reviews Sevilla.

El siguiente paso natural del proyecto será construir el módulo de normalización de platos, para agrupar variantes como:

- `burger`, `burgers`, `cheeseburger`
- `taco`, `tacos`, `tuna tacos`
- `fries`, `french fries`, `home fries`
- `ice cream`, `frozen custard`

Esto permitirá pasar de menciones textuales a platos normalizados y preparar el ranking Hidden Gems.